# Phân Loại Trái Cây & Rau Củ Bằng ResNet50
### Image Classification | Transfer Learning

---

## Giới Thiệu
Notebook này tập trung vào việc sử dụng mô hình **ResNet50** để phân loại 36 loại trái cây và rau củ. Đây là kiến trúc tối ưu nhất cho bộ dữ liệu này, cân bằng giữa độ chính xác và chi phí tính toán.

### Chiến lược huấn luyện:
- **Phase 1: Feature Extraction**: Đóng băng ResNet50, chỉ huấn luyện các lớp phân loại mới thêm vào.
- **Phase 2: Fine-tuning**: Mở khóa một số lớp cuối của ResNet50 để tinh chỉnh mô hình sâu hơn với dữ liệu thực phẩm.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

from sklearn.metrics import confusion_matrix, classification_report, f1_score

# Cấu hình
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS_PHASE1 = 10
EPOCHS_PHASE2 = 20
SEED         = 42

BASE_DIR    = './dataset'
TRAIN_DIR   = os.path.join(BASE_DIR, 'train')
VAL_DIR     = os.path.join(BASE_DIR, 'validation')
TEST_DIR    = os.path.join(BASE_DIR, 'test')

print('TensorFlow version:', tf.__version__)
gpu = tf.config.list_physical_devices('GPU')
print('GPU available:', gpu if gpu else '(Using CPU)')

## Tải & Chuẩn Bị Dữ Liệu

In [ ]:
# Tải dataset từ Kaggle (Cần file kaggle.json)
from google.colab import files
if not os.path.exists('kaggle.json'):
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d kritikseth/fruit-and-vegetable-image-recognition
!unzip -q fruit-and-vegetable-image-recognition.zip -d ./dataset

print("Dataset đã sẵn sàng!")

In [ ]:
# Data Augmentation tối ưu cho ResNet
train_datagen = ImageDataGenerator(
    preprocessing_function=resnet_preprocess,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(preprocessing_function=resnet_preprocess)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', seed=SEED
)
val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', seed=SEED, shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', seed=SEED, shuffle=False
)

class_names = list(train_gen.class_indices.keys())
NUM_CLASSES = len(class_names)
print(f"Đã tải {NUM_CLASSES} lớp.")

## Xây Dựng Mô Hình

In [ ]:
def build_resnet_model(num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    base_model.trainable = False # Khóa các lớp tiền huấn luyện

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='Fruit_ResNet50')
    return model, base_model

model, base_model = build_resnet_model(NUM_CLASSES)
model.summary()

## Huấn Luyện

In [ ]:
# --- Phase 1: Feature Extraction ---
print("Starting Phase 1: Feature Extraction...")
model.compile(optimizer=optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

history_p1 = model.fit(
    train_gen, epochs=EPOCHS_PHASE1,
    validation_data=val_gen,
    callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)

# --- Phase 2: Fine-tuning ---
print("\nStarting Phase 2: Fine-tuning...")
base_model.trainable = True
# Chỉ mở khóa 30 lớp cuối để tránh phá vỡ trọng số tốt
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

res_callbacks = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(factor=0.2, patience=3, min_lr=1e-7),
    callbacks.ModelCheckpoint('best_fruit_model.h5', save_best_only=True)
]

history_p2 = model.fit(
    train_gen, epochs=EPOCHS_PHASE2,
    validation_data=val_gen,
    callbacks=res_callbacks
)

## Đánh Giá

In [ ]:
# Visualize kết quả huấn luyện
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.title('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.title('Loss')
    plt.legend()
    plt.show()

plot_history(history_p2)

In [ ]:
# Đánh giá trên tập test
test_loss, test_acc = model.evaluate(test_gen)
print(f"\nTest Accuracy: {test_acc*100:.2f}%")

# Confusion Matrix
test_gen.reset()
y_pred = np.argmax(model.predict(test_gen), axis=1)
y_true = test_gen.classes

plt.figure(figsize=(15, 12))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.show()

## Lưu Mô Hình

In [ ]:
model.save('fruit_resnet50_final.h5')
with open('fruit_classes.json', 'w') as f:
    json.dump(class_names, f)
print("Model and classes saved successfully!")